# 01 - SQL exploration with DuckDB

**Goal:** load the raw CSV with SQL, confirm data quality, and build the core summary tables.

DuckDB runs SQL directly on the CSV file, inside Python, with no database server to install.
Every query lives in its own file under `sql/`, the way an analytics team would keep them.
The queries use aggregations, CTEs, window functions and joins.

In [1]:
import duckdb                      # DuckDB lets us run SQL on a CSV file without installing a database server
import pandas as pd                # pandas shows query results as tables
from pathlib import Path           # Path builds file paths that work on Windows, Mac and Linux

pd.set_option("display.max_columns", None)   # never hide columns when printing tables
pd.set_option("display.width", 200)          # allow wide tables to print on one line

In [2]:
ROOT = Path.cwd()                  # start from the folder this notebook is running in
if ROOT.name == "notebooks":       # if we are inside the notebooks folder...
    ROOT = ROOT.parent             # ...move up one level to the project root
DATA_FILE = ROOT / "data" / "raw" / "cookie_cats.csv"   # full path to the raw CSV
SQL_DIR = ROOT / "sql"             # folder that holds our .sql files
print("Project root:", ROOT)       # print so you can confirm the path is correct
print("CSV exists:", DATA_FILE.exists())   # should print True

Project root: D:\Analytics\CookieCats-A-B-Test
CSV exists: True


**Checkpoint:** `CSV exists: True`. If it prints `False`, the file is not in `data/raw/`.

In [3]:
con = duckdb.connect()             # open an in-memory DuckDB database (nothing is saved to disk)
csv_path = DATA_FILE.as_posix()    # convert the path to forward slashes so SQL accepts it on Windows too
con.execute(f"CREATE OR REPLACE VIEW players AS SELECT * FROM read_csv_auto('{csv_path}')")  # a view = a saved query we can SELECT from
print(con.execute("DESCRIBE players").df())   # show column names and the types DuckDB detected

      column_name column_type null   key default extra
0          userid      BIGINT  YES  None    None  None
1         version     VARCHAR  YES  None    None  None
2  sum_gamerounds      BIGINT  YES  None    None  None
3     retention_1     BOOLEAN  YES  None    None  None
4     retention_7     BOOLEAN  YES  None    None  None


**Checkpoint:** five columns - `userid BIGINT`, `version VARCHAR`, `sum_gamerounds BIGINT`,
`retention_1 BOOLEAN`, `retention_7 BOOLEAN`.

In [4]:
sql_files = sorted(SQL_DIR.glob("*.sql"))     # list all .sql files, sorted so 01 runs before 02
results = {}                                  # empty dictionary to keep each result table
for sql_file in sql_files:                    # loop over the files one by one
    query = sql_file.read_text()              # read the SQL text from the file
    table = con.execute(query).df()           # run the query and convert the result to a pandas table
    results[sql_file.stem] = table            # store the table using the file name (without .sql) as the key
    print("=" * 80)                           # a divider line to keep the output readable
    print(sql_file.name)                      # which query this is
    print(table.to_string(index=False))       # print the full table without the pandas row index

01_data_quality_checks.sql
 total_rows  distinct_players  duplicate_rows  null_userid  null_version  null_gamerounds  null_retention_1  null_retention_7  min_rounds  max_rounds  zero_round_players
      90189             90189               0            0             0                0                 0                 0           0       49854                3994
02_group_sizes.sql
version  players  share_pct
gate_30    44700     49.563
gate_40    45489     50.437
03_retention_by_version.sql
version  players  d1_retained  d1_retention_pct  d7_retained  d7_retention_pct  mean_rounds  median_rounds
gate_30    44700        20034             44.82         8502             19.02        52.46           17.0
gate_40    45489        20119             44.23         8279             18.20        51.30           16.0


04_engagement_distribution.sql
version  rank_in_group  userid  sum_gamerounds
gate_30              1 6390605           49854
gate_30              2  871500            2961
gate_30              3 4832608            2438
gate_30              4 5133952            2251
gate_30              5 9640085            2156
gate_40              1 3271615            2640
gate_40              2 5346171            2294
gate_40              3 4090246            2124
gate_40              4 9791599            2063
gate_40              5  725080            2015
05_percentiles.sql
version  p25_rounds  p50_rounds  p75_rounds  p90_rounds  p99_rounds  max_rounds
gate_30         5.0        17.0        50.0       135.0      493.00       49854
gate_40         5.0        16.0        52.0       134.0      492.12        2640
06_rounds_buckets_descriptive.sql
    rounds_bucket version  players  share_of_group_pct  d7_retention_pct
      1) 0 rounds gate_30     1937                4.33              0.83
      1) 0 ro


07_retention_consistency.sql
 retention_1  retention_7  players  share_pct
       False        False    46437      51.49
       False         True     3599       3.99
        True        False    26971      29.90
        True         True    13182      14.62
08_lift_vs_control_with_join.sql
     role  gate_level version  players  d1_pct  d1_diff_vs_control_pp  d7_pct  d7_diff_vs_control_pp  d7_relative_lift_pct
  control          30 gate_30    44700   44.82                  0.000   19.02                   0.00                  0.00
treatment          40 gate_40    45489   44.23                 -0.591   18.20                  -0.82                 -4.31


**What the eight queries say**

- **01** The data is structurally clean: 90,189 rows, 90,189 distinct players, no duplicates, no nulls.
  Rounds run from 0 to 49,854 and 3,994 players installed without ever playing a round.
- **02** The groups are not exactly equal: gate_30 has 44,700 players (49.563%) and gate_40 has 45,489 (50.437%).
  Notebook 03 tests whether that gap is suspicious.
- **03** Retention is lower in gate_40 on both days (D1 44.82% vs 44.23%, D7 19.02% vs 18.20%).
  Notebook 04 tests whether that could be chance.
- **04** One gate_30 player logged 49,854 rounds, about 17 times the next highest player in their group (2,961).
- **05** The medians (17 vs 16) and the 90th/99th percentiles are almost identical across groups.
- **06** Retention rises steeply with rounds played - but rounds happen *after* assignment,
  so this table describes players and must not be used to compare versions (proved in notebook 07).
- **07** 3,599 players (3.99%) skipped day 1 and still came back on day 7, so D1 and D7 are related but separate behaviors.
- **08** gate_40 is 0.591 pp lower on D1 and 0.820 pp lower on D7, a 4.31% relative drop in D7.

In [5]:
summary_sql = results["03_retention_by_version"]                        # grab the core A/B summary table
out_path = ROOT / "data" / "processed" / "sql_retention_summary.csv"    # where to save it
out_path.parent.mkdir(parents=True, exist_ok=True)                      # create the folder if it does not exist
summary_sql.to_csv(out_path, index=False)                               # save without the row index
print("Saved:", out_path)                                               # confirm the save
con.close()                                                             # close the DuckDB connection

Saved: D:\Analytics\CookieCats-A-B-Test\data\processed\sql_retention_summary.csv
